# otro_pipe/03 — clustering jerarquico con DTW + dendrograma (exploratorio)

Notebook exploratorio, no una grilla de produccion: arma un dendrograma
sobre una MUESTRA de pares cliente-producto usando distancia DTW, para que
puedas mirar las formas de los clusters y elegir vos misma cuantos/cuales
usar -- no elige "el mejor k" sola.

Reusa el motor DTW de `pipe_nuevo/05_DTW_clusters.ipynb` (distancia con
`dtaidistance`, banda de Sakoe-Chiba, DBA para el centroide) y agrega:

- **Escalado `'rango'`** (min-max), para comparar contra `'zscore'`.
- **Normalizacion de la distancia por el largo del camino de alineacion**:
  DTW acumula mas costo cuanto mas larga es la serie -- sin esto, series
  largas parecen "mas lejos de todo" solo por ser largas, no por tener una
  forma distinta. Se divide el costo DTW por la cantidad de pasos del
  warping path.
- **Clustering JERARQUICO** (`scipy.cluster.hierarchy`, linkage promedio
  sobre la matriz de distancia precalculada) en vez de k-means -- el
  k-means de `05_` no puede dar un dendrograma porque no arma una
  jerarquia, solo particiones planas.
- Por cada cluster (para varios cortes de k candidatos): la forma tanto
  como **centroide DBA** (promedio sintetico) como **medoide** (el miembro
  real mas central) -- pediste ver las dos.

Como el dendrograma necesita la matriz de distancia COMPLETA (O(n^2)), esto
corre sobre una MUESTRA (los pares de mayor volumen), no sobre todos los
pares -- para eso esta `05_DTW_clusters.ipynb`, que si escala a cientos de
miles via k-means.


## 0) Setup


In [ ]:
import gc, json, os, time
from pathlib import Path

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

from dtaidistance import dtw
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform


def resolver_bucket() -> Path:
    """VM de la catedra -> ~/buckets/b1 | Colab -> /content/buckets/b1 | local ultimo."""
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1",
                 "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError("No encontre el bucket. Defini LABO3_BUCKET.")


BUCKET   = resolver_bucket()
DIR_RAW  = BUCKET / "datasets"
RUTA_FE  = BUCKET / "datasets_fe"          # de aca lee (cache de FE de pipe_nuevo/otro_pipe)
DIR_RUNS = BUCKET / "exp_dtw_dendrograma"
DIR_RUNS.mkdir(parents=True, exist_ok=True)

SERIE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
         "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
TINTA, TINTA2, MUDO = "#0b0b0b", "#52514e", "#898781"
GRILLA, EJE_C, FONDO = "#e1e0d9", "#c3c2b7", "#fcfcfb"

plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO,
    "axes.edgecolor": EJE_C, "axes.labelcolor": TINTA2,
    "text.color": TINTA, "xtick.color": MUDO, "ytick.color": MUDO,
    "grid.color": GRILLA, "grid.linewidth": .8,
    "axes.grid": True, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 9, "axes.titlesize": 10, "figure.dpi": 110,
    "legend.frameon": False,
})


def limpiar(ax, titulo=None, y=None, x=None):
    if titulo:
        ax.set_title(titulo, color=TINTA, loc="left", pad=10)
    if y:
        ax.set_ylabel(y)
    if x:
        ax.set_xlabel(x)
    ax.grid(axis="x", visible=False)
    return ax


def guardar(fig, nombre, subcarpeta=None, mostrar=True):
    d = DIR_RUN if subcarpeta is None else DIR_RUN / subcarpeta
    d.mkdir(parents=True, exist_ok=True)
    p = d / f"{nombre}.png"
    fig.savefig(p, dpi=140, bbox_inches="tight", facecolor=FONDO)
    print(f"   [fig] {p.relative_to(BUCKET)}")
    plt.show() if mostrar else plt.close(fig)
    return p


_C_OK = dtw.try_import_c()
print(f"BUCKET: {BUCKET}")
print(f"DTW con backend C: {_C_OK}")
if not _C_OK:
    print("   ATENCION: sin backend C esto puede ser lento. pip install -U dtaidistance")

try:
    dtw.warping_path(np.zeros(5), np.zeros(5), window=2)
    _WP_WINDOW = True
except TypeError:
    _WP_WINDOW = False
print(f"warping_path acepta window: {_WP_WINDOW}")


In [ ]:
PARAM = {
    # ── de donde salen las series (igual que 05_DTW_clusters) ────────────
    'fuente': 'preprocesado',
    'archivo_preprocesado': None,
    'densificar': 'desde_nacimiento',
    'mes_corte': 201906,   # corte anti-leakage: no mirar meses de val/test/futuro
    'min_meses': 18,

    # ── muestra: primero los PRODUCTOS mas importantes, despues sus pares ──
    # (el dendrograma necesita la matriz O(n^2) completa, por eso el techo)
    'top_productos': 50,     # productos de mayor volumen total
    'muestra_pares': 2000,   # techo de pares (de esos productos) para la matriz O(n^2)

    # ── grilla a explorar: escalado x ventana ─────────────────────────────
    'escalados': ('zscore', 'rango'),
    'ventanas': (12, 18),
    'normalizar_por_largo': True,   # divide el costo DTW por el largo del warping path

    # ── clustering jerarquico ──────────────────────────────────────────────
    'metodo_linkage': 'average',   # 'average' | 'complete' (no 'ward': pide euclideana)
    'lista_k_explorar': (3, 4, 5, 6, 8),
    'muestra_silhouette': 300,

    'semilla': 102191,
}

ESCALADOS_VALIDOS = ('media', 'zscore', 'maximo', 'ninguno', 'rango')
for e in PARAM['escalados']:
    if e not in ESCALADOS_VALIDOS:
        raise ValueError(f"escalado invalido: {e!r}. Opciones: {ESCALADOS_VALIDOS}")
if PARAM['metodo_linkage'] not in ('average', 'complete', 'single'):
    raise ValueError(f"metodo_linkage invalido: {PARAM['metodo_linkage']!r} "
                     f"('ward' no sirve con distancias no-euclideanas como DTW)")

RNG = np.random.default_rng(PARAM['semilla'])
CATS = ["cat1", "cat2", "cat3", "brand"]

print(f"escalados a explorar: {PARAM['escalados']}")
print(f"ventanas a explorar : {PARAM['ventanas']}")
print(f"normalizar por largo: {PARAM['normalizar_por_largo']}")
print(f"muestra_pares       : {PARAM['muestra_pares']}")


## 1) Panel producto-cliente-mes (portado de `05_DTW_clusters.ipynb`)


In [ ]:
t0 = time.time()


def a_m(p):
    """AAAAMM -> indice de mes continuo, para poder sumar y restar meses."""
    return (p // 100) * 12 + (p % 100)


def m_a_periodo(m):
    return ((m - 1) // 12) * 100 + ((m - 1) % 12) + 1


def normalizar_periodo(df: pl.DataFrame) -> pl.DataFrame:
    """Deja 'periodo' como Int64 AAAAMM, venga como Date, string o entero."""
    dt = df.schema['periodo']
    try:
        temporal = dt.is_temporal()
    except AttributeError:
        temporal = dt in (pl.Date, pl.Datetime)
    if temporal:
        return df.with_columns(
            (pl.col('periodo').dt.year() * 100 + pl.col('periodo').dt.month())
            .cast(pl.Int64).alias('periodo'))
    if dt == pl.Utf8:
        return df.with_columns(
            pl.col('periodo').str.replace_all(r'\D', '').str.slice(0, 6)
              .cast(pl.Int64).alias('periodo'))
    return df.with_columns(pl.col('periodo').cast(pl.Int64))


if PARAM['fuente'] == 'preprocesado':
    disp = sorted(RUTA_FE.glob("features_sin_escalar_*.parquet"))
    if not disp:
        raise FileNotFoundError(f"No hay features_sin_escalar_*.parquet en {RUTA_FE}. "
                                f"Corre el preprocesamiento/FE primero, o usa fuente='crudo'.")
    if PARAM['archivo_preprocesado']:
        path_pre = RUTA_FE / PARAM['archivo_preprocesado']
        if not path_pre.exists():
            raise FileNotFoundError(f"No existe {path_pre}.\nDisponibles: {[p.name for p in disp]}")
    else:
        path_pre = max(disp, key=lambda p: p.stat().st_mtime)
    print(f"Leyendo {path_pre.name}")

    raw = pl.read_parquet(path_pre)
    if "tn0" in raw.columns and "tn" not in raw.columns:
        raw = raw.rename({"tn0": "tn"})
    faltan = [c for c in ["product_id", "customer_id", "periodo", "tn"] if c not in raw.columns]
    if faltan:
        raise ValueError(f"El parquet no tiene {faltan}. Columnas: {raw.columns}")
    raw = normalizar_periodo(raw)
    cats_ok = [c for c in CATS if c in raw.columns]
    panel = (raw.group_by(["product_id", "customer_id", "periodo"])
                .agg(pl.col("tn").sum().alias("tn"))
                .join(raw.select(["product_id"] + cats_ok).unique(subset=["product_id"]),
                      on="product_id", how="left"))
else:
    sell = normalizar_periodo(pl.read_csv(DIR_RAW / "sell-in.txt.gz", separator="\t"))
    prod = (pl.read_csv(DIR_RAW / "tb_productos.txt", separator="\t")
              .unique(subset=["product_id"]))
    cats_ok = [c for c in CATS if c in prod.columns]
    panel = (sell.group_by(["product_id", "customer_id", "periodo"])
                 .agg(pl.col("tn").sum().alias("tn"))
                 .join(prod.select(["product_id"] + cats_ok), on="product_id", how="left"))

panel = panel.with_columns(a_m(pl.col("periodo")).alias("m"))

if PARAM['mes_corte'] is not None:
    m_corte = a_m(PARAM['mes_corte'])
    antes = panel.height
    panel = panel.filter(pl.col("m") < m_corte)
    print(f"Corte anti-leakage en {PARAM['mes_corte']}: {antes:,} -> {panel.height:,} filas")

M_MIN, M_MAX = int(panel["m"].min()), int(panel["m"].max())
print(f"panel: {panel.height:,} filas · "
     f"{panel['product_id'].n_unique()} productos x {panel['customer_id'].n_unique()} clientes")
print(f"meses {m_a_periodo(M_MIN)} -> {m_a_periodo(M_MAX)}  ({M_MAX - M_MIN + 1} meses)")
print(f"[{time.time()-t0:.0f}s]")


## 2) Densificacion + muestra + series escaladas

Igual criterio que `05_DTW_clusters.ipynb`: `m_nace`/`m_ultima` se calculan
SOLO con ventas reales (`tn > 0`), no con las filas de cero del cartesiano
-- si no, la serie "naceria" en el momento en que cliente y producto
empezaron a coexistir, no cuando el par empezo a comprar de verdad.


In [ ]:
t0 = time.time()
KEYS = ["product_id", "customer_id"]

vida = (panel.filter(pl.col("tn") > 0)
             .group_by(KEYS)
             .agg(pl.col("m").min().alias("m_nace"),
                  pl.col("m").max().alias("m_ultima"),
                  pl.col("tn").sum().alias("tn_total"),
                  pl.len().alias("meses_con_venta")))

if PARAM['densificar'] == 'desde_nacimiento':
    vida = vida.with_columns(pl.lit(M_MAX).alias("m_fin"))
else:
    vida = vida.with_columns(pl.col("m_ultima").alias("m_fin"))
vida = vida.with_columns((pl.col("m_fin") - pl.col("m_nace") + 1).alias("largo"))

elegibles = vida.filter(pl.col("largo") >= PARAM['min_meses'])
print(f"pares: {vida.height:,} totales -> {elegibles.height:,} con >= "
     f"{PARAM['min_meses']} meses de serie")
if elegibles.height == 0:
    raise ValueError(f"Ningun par tiene >= {PARAM['min_meses']} meses de serie. "
                     f"Baja PARAM['min_meses'] o revisa PARAM['mes_corte'].")

# Primero los PRODUCTOS mas importantes (no pares sueltos de productos chicos
# que por casualidad tengan un cliente grande), despues sus pares.
top_productos = (elegibles.group_by("product_id").agg(pl.col("tn_total").sum().alias("tn_producto"))
                          .sort("tn_producto", descending=True)
                          .head(PARAM['top_productos'])["product_id"])
elegibles_top = elegibles.filter(pl.col("product_id").is_in(top_productos))
print(f"top {PARAM['top_productos']} productos por volumen -> {elegibles_top.height:,} pares "
     f"(de {elegibles.height:,} elegibles en total)")

# Muestra para el DENDROGRAMA (matriz O(n^2)): techo de pares, los de mayor volumen
# DENTRO de esos productos top. La extension a todos los pares de esos productos
# (aunque no hayan entrado en esta muestra) pasa en la seccion final.
if PARAM['muestra_pares'] and elegibles_top.height > PARAM['muestra_pares']:
    elegibles = elegibles_top.sort("tn_total", descending=True).head(PARAM['muestra_pares'])
else:
    elegibles = elegibles_top
print(f"muestra para el dendrograma: {elegibles.height:,} pares")

grilla = (elegibles.select(KEYS + ["m_nace", "m_fin"])
                   .with_columns(pl.int_ranges("m_nace", pl.col("m_fin") + 1).alias("m"))
                   .explode("m")
                   .select(KEYS + ["m"]))
denso = (grilla.join(panel.select(KEYS + ["m", "tn"]), on=KEYS + ["m"], how="left")
               .with_columns(pl.col("tn").fill_null(0.0))
               .sort(KEYS + ["m"]))

_ceros = int((denso["tn"] == 0).sum())
print(f"panel denso: {denso.height:,} filas ({_ceros:,} ceros = {100*_ceros/denso.height:.0f}%)")
print(f"[{time.time()-t0:.0f}s]")


In [ ]:
def escalar(v: np.ndarray, modo: str) -> np.ndarray:
    v = np.asarray(v, dtype=np.float64)
    if modo == 'media':
        mu = v.mean()
        out = v / mu if abs(mu) > 1e-9 else v
    elif modo == 'zscore':
        sd = v.std()
        out = (v - v.mean()) / sd if sd > 1e-9 else v - v.mean()
    elif modo == 'maximo':
        mx = np.abs(v).max()
        out = v / mx if mx > 1e-9 else v
    elif modo == 'rango':
        # min-max (NUEVO, pedido para comparar contra zscore): deja la serie
        # en [0, 1]. Distinto de 'maximo' (que solo divide por el pico, sin
        # correr el piso a 0).
        mn, mx = v.min(), v.max()
        rango = mx - mn
        out = (v - mn) / rango if rango > 1e-9 else v - mn
    elif modo == 'ninguno':
        out = v
    else:
        raise ValueError(f"escalado invalido: {modo!r}")
    return np.ascontiguousarray(out, dtype=np.float64)


def armar_series(denso_df: pl.DataFrame, modo: str):
    g = (denso_df.sort(KEYS + ["m"])
                 .group_by(KEYS, maintain_order=True)
                 .agg(pl.col("tn").alias("serie"), pl.col("m").min().alias("m0")))
    pares = list(zip(g["product_id"].to_list(), g["customer_id"].to_list()))
    crudas = [np.asarray(s, dtype=np.float64) for s in g["serie"].to_list()]
    escaladas = [escalar(s, modo) for s in crudas]
    return pares, escaladas, crudas, g["m0"].to_list()


PARES, _, CRUDAS, M0 = armar_series(denso, 'ninguno')
LARGOS = np.array([len(s) for s in CRUDAS])
print(f"{len(CRUDAS):,} series   largo min/mediana/max: "
     f"{LARGOS.min()} / {int(np.median(LARGOS))} / {LARGOS.max()}")


## 3) Motor DTW — banda, distancia normalizada por largo, DBA, medoide

`d_dtw()` con `normalizar_largo=True` divide el costo DTW por la cantidad
de pasos del warping path -- sin esto, dos series de formas parecidas pero
largos MUY distintos (una recien nacida, otra con 3 años de vida) quedan
mas "lejos" solo por acumular mas costo, no por ser realmente distintas.


In [ ]:
def banda(a, b, window):
    """Banda de Sakoe-Chiba factible: se ensancha lo justo si los largos
    difieren mas que 'window', para no devolver inf en pares que nacieron
    en meses distintos."""
    if window is None:
        return None
    return max(int(window), abs(len(a) - len(b)))


def d_dtw(a, b, window=None, normalizar_largo=False):
    w = banda(a, b, window)
    costo = dtw.distance_fast(a, b, window=w, use_pruning=False)
    if normalizar_largo:
        path = dtw.warping_path(a, b, window=w) if _WP_WINDOW else dtw.warping_path(a, b)
        costo = costo / len(path) if path else costo
    return costo


def matriz_dtw(series, window=None, normalizar_largo=False):
    """Matriz simetrica completa. Solo para muestras chicas: es O(n^2)."""
    n = len(series)
    D = np.zeros((n, n), dtype=np.float64)
    for i in range(n):
        si = series[i]
        for j in range(i + 1, n):
            D[i, j] = D[j, i] = d_dtw(si, series[j], window, normalizar_largo)
    mal = ~np.isfinite(D)
    if mal.any():
        fin = D[~mal]
        D[mal] = (fin.max() * 10.0) if fin.size else 1.0
        print(f"   aviso: {int(mal.sum())} distancias no finitas saneadas")
    return D


def dba(miembros, centro, window=None, iters=3):
    """Centroide sintetico (DTW Barycenter Averaging): promedia, en cada
    paso, los valores de cada miembro ALINEADOS al centroide actual."""
    centro = np.ascontiguousarray(centro, dtype=np.float64)
    if not miembros:
        return centro
    T = len(centro)
    for _ in range(iters):
        acum = np.zeros(T, dtype=np.float64)
        cuenta = np.zeros(T, dtype=np.float64)
        for s in miembros:
            w = banda(centro, s, window)
            path = (dtw.warping_path(centro, s, window=w) if _WP_WINDOW
                    else dtw.warping_path(centro, s))
            for i, j in path:
                acum[i] += s[j]
                cuenta[i] += 1.0
        centro = np.where(cuenta > 0, acum / np.maximum(cuenta, 1.0), centro)
        centro = np.ascontiguousarray(centro, dtype=np.float64)
    return centro


def medoide(idx_miembros, D):
    """El miembro real (no sintetico) que minimiza la distancia total a los
    demas del mismo cluster -- la serie 'mas central' de verdad, para
    comparar contra el centroide DBA (que es un promedio, no existe en los
    datos)."""
    if len(idx_miembros) == 1:
        return idx_miembros[0]
    sub = D[np.ix_(idx_miembros, idx_miembros)]
    return idx_miembros[int(sub.sum(axis=1).argmin())]


print("motor DTW listo (banda, distancia normalizada por largo, DBA, medoide)")


## 4) Grilla escalado x ventana — dendrograma + forma por cluster

Por cada combinacion: matriz de distancia completa, dendrograma (linkage
sobre esa matriz), y para cada `k` candidato de `lista_k_explorar` la forma
de cada cluster (DBA + medoide) y las etiquetas guardadas -- vos elegis
despues cual combinacion/k usar, esto no elige por vos.


In [ ]:
def plot_forma_por_cluster(SERIES, labels0, k, centroides_dba, medoides, titulo):
    orden_cl = sorted(centroides_dba.keys())
    ncol = min(3, len(orden_cl))
    nfil = int(np.ceil(len(orden_cl) / ncol))
    fig, axes = plt.subplots(nfil, ncol, figsize=(4.6 * ncol, 3.0 * nfil), squeeze=False)
    for pos, cl in enumerate(orden_cl):
        ax = axes[pos // ncol][pos % ncol]
        miembros = np.flatnonzero(labels0 == cl)
        cen = centroides_dba[cl]
        med = SERIES[medoides[cl]]
        T = max(len(cen), int(np.median([len(SERIES[i]) for i in miembros])))
        sub = miembros if len(miembros) <= 60 else RNG.choice(miembros, 60, replace=False)
        Mser = np.full((len(sub), T), np.nan)
        for r, i in enumerate(sub):
            v = SERIES[i][:T]
            Mser[r, :len(v)] = v
            ax.plot(np.arange(len(v)), v, color=GRILLA, linewidth=.7, zorder=1)
        with np.errstate(all='ignore'):
            p10 = np.nanpercentile(Mser, 10, axis=0)
            p90 = np.nanpercentile(Mser, 90, axis=0)
        ax.fill_between(np.arange(T), p10, p90, color=SERIE[pos % len(SERIE)],
                        alpha=.18, zorder=2, linewidth=0)
        ax.plot(np.arange(len(cen)), cen, color=SERIE[pos % len(SERIE)],
               linewidth=2.2, zorder=3, label='DBA')
        ax.plot(np.arange(len(med)), med, color=TINTA, linewidth=1.3,
               linestyle='--', zorder=3, label='medoide')
        ax.legend(fontsize=6, loc='upper right')
        limpiar(ax, f"cl {cl} — {len(miembros)} pares", "valor escalado", "mes desde el inicio")
    for pos in range(len(orden_cl), nfil * ncol):
        axes[pos // ncol][pos % ncol].axis("off")
    fig.suptitle(f"{titulo} — forma por cluster (banda=percentil 10-90, "
                f"linea=DBA, punteada=medoide)", color=TINTA2, fontsize=8.5, y=1.01)
    fig.tight_layout()
    guardar(fig, f"forma_k{k}", mostrar=False)


filas_comparacion = []

for escalado in PARAM['escalados']:
    _, SERIES, _, _ = armar_series(denso, escalado)

    for ventana in PARAM['ventanas']:
        SLUG = f"esc{escalado}_w{ventana}" + ("_normL" if PARAM['normalizar_por_largo'] else "")
        DIR_RUN = DIR_RUNS / SLUG
        DIR_RUN.mkdir(parents=True, exist_ok=True)
        print(f"\n{'='*74}\n{SLUG}\n{'='*74}")

        t0 = time.time()
        D = matriz_dtw(SERIES, window=ventana, normalizar_largo=PARAM['normalizar_por_largo'])
        print(f"matriz de distancia: {D.shape}   [{time.time()-t0:.0f}s]")

        Z = linkage(squareform(D, checks=False), method=PARAM['metodo_linkage'])
        np.save(DIR_RUN / "linkage_Z.npy", Z)
        np.save(DIR_RUN / "matriz_distancia.npy", D)

        fig, ax = plt.subplots(figsize=(14, 5))
        dendrogram(Z, ax=ax, no_labels=True,
                  color_threshold=0.7 * float(Z[:, 2].max()))
        limpiar(ax, f"{SLUG} — dendrograma (linkage {PARAM['metodo_linkage']})",
               "distancia DTW" + (" (normalizada por largo)" if PARAM['normalizar_por_largo'] else ""),
               "pares (una hoja = un par cliente-producto)")
        fig.tight_layout()
        guardar(fig, "dendrograma", mostrar=False)

        etiquetas_combo = pl.DataFrame({
            'product_id': [p for p, _ in PARES],
            'customer_id': [c for _, c in PARES],
        })

        for k in PARAM['lista_k_explorar']:
            labels0 = fcluster(Z, k, criterion='maxclust') - 1
            etiquetas_combo = etiquetas_combo.with_columns(pl.Series(f'cluster_k{k}', labels0))

            tam = np.bincount(labels0, minlength=labels0.max() + 1)
            k_efectivo = int((tam > 0).sum())
            sil = (float(silhouette_score(D, labels0, metric='precomputed'))
                  if k_efectivo >= 2 else float('nan'))

            centroides_dba, medoides = {}, {}
            for j in range(labels0.max() + 1):
                miembros_idx = np.flatnonzero(labels0 == j)
                if len(miembros_idx) == 0:
                    continue
                miembros_series = [SERIES[i] for i in miembros_idx]
                largo_centro = int(np.median([len(s) for s in miembros_series]))
                i_centro = int(np.argmin([abs(len(s) - largo_centro) for s in miembros_series]))
                centroides_dba[j] = dba(miembros_series, miembros_series[i_centro], window=ventana)
                medoides[j] = medoide(miembros_idx.tolist(), D)

            print(f"  k={k:2d}  k_efectivo={k_efectivo}  tam={tam.tolist()}  silhouette={sil:+.4f}")
            plot_forma_por_cluster(SERIES, labels0, k, centroides_dba, medoides, f"{SLUG}, k={k}")
            # Se guardan los centroides DBA de este (combo, k) para poder reusarlos
            # despues en la seccion de extension a todos los pares, sin recalcular.
            np.savez(DIR_RUN / f"centroides_dba_k{k}.npz",
                    **{f"cluster{j}": v for j, v in centroides_dba.items()})

            filas_comparacion.append({
                'escalado': escalado, 'ventana': ventana, 'k': k, 'k_efectivo': k_efectivo,
                'tam_min': int(tam.min()), 'tam_max': int(tam.max()), 'silhouette': round(sil, 4),
            })

        etiquetas_combo.write_parquet(DIR_RUN / "etiquetas_todos_los_k.parquet")
        print(f"  guardado: {DIR_RUN.relative_to(BUCKET)}/etiquetas_todos_los_k.parquet, "
             f"linkage_Z.npy, matriz_distancia.npy, dendrograma.png, forma_k*.png")
        gc.collect()

tabla_comparacion = pl.DataFrame(filas_comparacion).sort(['escalado', 'ventana', 'k'])
tabla_comparacion.write_csv(DIR_RUNS / "comparacion_grilla.csv")
print(f"\n{'='*74}\ncomparacion_grilla.csv ({tabla_comparacion.height} filas):")
print(tabla_comparacion)


## 5) Extender la combinacion elegida a TODOS los pares de los productos top

Mira los graficos de arriba, elegi una combinacion (escalado, ventana, k),
completa `PARAM['combo_elegido']`, y corre esta seccion: asigna CADA par de
los `top_productos` (no solo los `muestra_pares` que entraron en el
dendrograma) al centroide DBA mas cercano -- mismo mecanismo `asignar()`
que usa el k-means de `05_DTW_clusters.ipynb`, pero arrancando de las
formas que ya elegiste a mano en vez de que k-means las descubra de cero.
Guarda `clusters_pc_dendrograma_*.parquet` con el mismo formato que
`05_DTW_clusters.ipynb` (columna `cluster_pc_k{K}`), para que cualquier
notebook que ya sepa leer esa convencion lo use sin cambios.


In [ ]:
PARAM['combo_elegido'] = {'escalado': 'zscore', 'ventana': 12, 'k': 6}

_ce = PARAM['combo_elegido']
_slug_elegido = (f"esc{_ce['escalado']}_w{_ce['ventana']}"
               + ("_normL" if PARAM['normalizar_por_largo'] else ""))
_dir_elegido = DIR_RUNS / _slug_elegido
_path_centroides = _dir_elegido / f"centroides_dba_k{_ce['k']}.npz"
if not _path_centroides.exists():
    raise FileNotFoundError(
        f"No encontre {_path_centroides}. PARAM['combo_elegido'] tiene que ser una "
        f"combinacion que ya corrio la seccion 4 (mira comparacion_grilla.csv para las "
        f"disponibles)."
    )
_npz = np.load(_path_centroides)
CENTROIDES_ELEGIDOS = {int(k.replace('cluster', '')): _npz[k] for k in _npz.files}
print(f"combo elegido: {_ce}   ({len(CENTROIDES_ELEGIDOS)} centroides cargados de {_path_centroides.name})")


In [ ]:
t0 = time.time()

# Densificado de TODOS los pares elegibles de los top_productos (no solo los
# muestra_pares que entraron en el dendrograma).
grilla_todos = (elegibles_top.select(KEYS + ["m_nace", "m_fin"])
                             .with_columns(pl.int_ranges("m_nace", pl.col("m_fin") + 1).alias("m"))
                             .explode("m")
                             .select(KEYS + ["m"]))
denso_todos = (grilla_todos.join(panel.select(KEYS + ["m", "tn"]), on=KEYS + ["m"], how="left")
                          .with_columns(pl.col("tn").fill_null(0.0))
                          .sort(KEYS + ["m"]))
print(f"panel denso (todos los pares de los top {PARAM['top_productos']} productos): "
     f"{denso_todos.height:,} filas   [{time.time()-t0:.0f}s]")

PARES_TODOS, SERIES_TODOS, _, _ = armar_series(denso_todos, _ce['escalado'])
print(f"{len(PARES_TODOS):,} pares a asignar (vs {len(PARES):,} que entraron en el dendrograma)")

t0 = time.time()
_centros_lista = [CENTROIDES_ELEGIDOS[j] for j in sorted(CENTROIDES_ELEGIDOS)]
_orden_clusters = sorted(CENTROIDES_ELEGIDOS)
D_asignacion = np.empty((len(SERIES_TODOS), len(_centros_lista)), dtype=np.float64)
for jj, cen in enumerate(_centros_lista):
    cen = np.ascontiguousarray(cen, dtype=np.float64)
    for ii, s in enumerate(SERIES_TODOS):
        D_asignacion[ii, jj] = d_dtw(s, cen, _ce['ventana'], PARAM['normalizar_por_largo'])
_mal = ~np.isfinite(D_asignacion)
if _mal.any():
    _fin = D_asignacion[~_mal]
    D_asignacion[_mal] = (_fin.max() * 10.0) if _fin.size else 1.0
    print(f"  aviso: {int(_mal.sum())} distancias no finitas saneadas")

etiquetas_finales = np.array(_orden_clusters)[D_asignacion.argmin(axis=1)]
print(f"asignacion terminada.   [{time.time()-t0:.0f}s]")
print(f"tamaño de cada cluster: {dict(zip(*np.unique(etiquetas_finales, return_counts=True)))}")

COL_CLUSTER_OUT = f"cluster_pc_k{_ce['k']}"
clusters_finales = pl.DataFrame({
    'product_id':  [p for p, _ in PARES_TODOS],
    'customer_id': [c for _, c in PARES_TODOS],
    COL_CLUSTER_OUT: etiquetas_finales.astype(np.int32),
})

_nombre_out = f"clusters_pc_dendrograma_{_slug_elegido}_k{_ce['k']}.parquet"
clusters_finales.write_parquet(RUTA_FE / _nombre_out)
print(f"\nGuardado: {RUTA_FE / _nombre_out}")
print(f"  {clusters_finales.height:,} pares con cluster asignado (de los top {PARAM['top_productos']} productos)")
